In [ ]:
import pandas as pd
import sys
import nbformat
import numpy as np
import plotly.express as px

### Carregando o DataFrame ###

In [ ]:
file_path = 'social_media_mental_health.csv'

df = pd.read_csv(file_path)
df.head()

### Realizando a análise do DataFrame ###

In [ ]:
df.info()

In [ ]:
df.describe()

### Análises de Ansiedade (GAD-7) ###

In [ ]:
fig = px.histogram(df, x="GAD_7_Score", nbins=22, title="Distribuição do Score de Ansiedade (GAD-7)")
fig.show()

In [ ]:
fig = px.histogram(df, x="GAD_7_Severity", 
                   category_orders={"GAD_7_Severity": ["Minimal", "Mild", "Moderate", "Severe"]},
                   title="Distribuição por Severidade de Ansiedade")
fig.show()

### Análises de Depressão (PHQ-9) ###

In [ ]:
fig = px.histogram(df, x="PHQ_9_Score", nbins=22, title="Distribuição do Score de Depressão (PHQ-9)")
fig.show()

In [ ]:
fig = px.histogram(df, x="PHQ_9_Severity", 
                   category_orders={"PHQ_9_Severity": ["None-Minimal", "Mild", "Moderate", "Moderately Severe","Severe"]},
                   title="Distribuição por Severidade de Depressão")
fig.show()

### Análise de Correlações ###

In [ ]:
import plotly.figure_factory as ff
import numpy as np

cols = ['Daily_Screen_Time_Hours', 'Sleep_Duration_Hours', 
        'Late_Night_Usage', 'Social_Comparison_Trigger',
        'GAD_7_Score', 'PHQ_9_Score']

corr = df[cols].corr().round(2)

fig = ff.create_annotated_heatmap(
    z=corr.values,
    x=cols,
    y=cols,
    annotation_text=corr.values,
    colorscale='RdBu_r',
    showscale=True
)
fig.update_layout(title="Matriz de Correlação")
fig.show()

In [ ]:
print(df.groupby("Social_Comparison_Trigger")["GAD_7_Score"].mean())

In [ ]:
medias = df.groupby("Social_Comparison_Trigger")["GAD_7_Score"].mean()
social_0 = medias[0]
social_1 = medias[1]
diferenca_social = social_1 - social_0
print(f"A diferença entre os usuários que consideram que o conteúdo assistido gera inveja/insegurança e os que não, é de {diferenca_social:.2f} pontos na escala GAD-7 (Ansiedade).")
print(f"Isso representa uma diferença de {((diferenca_social / social_0) * 100):.2f}% na média de ansiedade.")


In [ ]:
fig_1 = px.scatter(df, x="Daily_Screen_Time_Hours", y="GAD_7_Score",
                 color="Activity_Type", opacity=0.5,
                 trendline="ols",
                 title="Tempo de Tela vs Ansiedade")
fig_1.show()

fig_2 = px.scatter(df, x="Daily_Screen_Time_Hours", y="PHQ_9_Score",
                 color="Activity_Type", opacity=0.5,
                 trendline="ols",
                 title="Tempo de Tela vs Depressão")
fig_2.show()

print('*Obs: No gráfico acima, tenho uma limitação visual devido a variável GAD_7_Score e PHQ_9_Score serem discretas e não contínuas.')

In [ ]:
fig = px.box(df, x="Primary_Platform", y="GAD_7_Score",
             color="Primary_Platform",
             title="Ansiedade por Plataforma")
fig.show()

#### Analisando a linha do usuário com nível de ansiedade muito elevado da plataforma do Youtube, considerando ser um Outlier ####

In [ ]:
df[(df['Primary_Platform'] == 'YouTube') & (df['GAD_7_Score'] == 21)]

##### Usuário outlier no YouTube: feminino, 19 anos, Hyper-Connected, 7.63h/dia, GAD-7 = 21 (Severe), PHQ-9 = 8 (Mild) — padrão consistente com o arquétipo.

### Análise do IQR ###

In [ ]:
iqr_por_plataforma = df.groupby("Primary_Platform")["GAD_7_Score"].describe()
print(iqr_por_plataforma[["25%", "50%", "75%"]])

### Análise de Arquétipos de Usuários ###

In [ ]:
df['User_Archetype'].unique()

In [ ]:
fig_1 = px.box(df, x="User_Archetype", y="Daily_Screen_Time_Hours",
             color="User_Archetype",
             title="Tempo de Tela por Arquétipo de Usuário")
fig_1.show()

fig_2 = px.box(df, x="User_Archetype", y="GAD_7_Score",
             color="User_Archetype",
             title="Ansiedade por Arquétipo de Usuário")
fig_2.show()

fig_3 = px.box(df, x="User_Archetype", y="Sleep_Duration_Hours",
             color="User_Archetype",
             title="Sono por Arquétipo de Usuário")
fig_3.show()

### Análise por Tipo de Gênero

In [ ]:
fig_1 = px.box(df, x="Gender", y="Daily_Screen_Time_Hours",
             color="Gender",
             title="Tempo de Tela por Gênero")

fig_1.show()

fig_2 = px.box(df, x="Gender", y="GAD_7_Score",
                 color="Gender",
                 title="Nível de Ansiedade por Gênero")
fig_2.show()

fig_3 = px.box(df, x="Gender", y="PHQ_9_Score",
                 color="Gender",
                 title="Nível de Depressão por Gênero")
fig_3.show()

print('Os tipos de gêneros não apresentaram diferenças significativas em relação ao tempo de tela, ansiedade e depressão, com médias bastante semelhantes.')

### Análise por Idade

In [ ]:
fig = px.scatter(df, x="Age", y="GAD_7_Score",
                 color="Activity_Type", opacity=0.5,
                 trendline="ols",
                 title="Idade vs Ansiedade")
fig.show()

fig_2 = px.box(df, x="Age", y="GAD_7_Score",
                 color="Age",
                 title="Nível de Ansiedade por Idade")
fig_2.show()

fig_3 = px.box(df, x="Age", y="PHQ_9_Score",
                 color="Age",
                 title="Nível de Depressão por Idade")
fig_3.show()

print('As idades dos usuários não apresentaram diferenças significativas em relação ao tempo de tela, ansiedade e depressão, com médias bastante semelhantes.')

### Análise de uso após a Meia-Noite

In [ ]:
fig_1 = px.box(df, x="Late_Night_Usage", y="GAD_7_Score",
                 color="Late_Night_Usage",
                 title="Uso Após a Meia-Noite vs Ansiedade")
fig_1.show()

fig_2 = px.box(df, x="Late_Night_Usage", y="PHQ_9_Score",
                 color="Late_Night_Usage",
                 title="Uso Após a Meia-Noite vs Depressão")
fig_2.show()

print('Usuários que relataram usar as redes sociais após a meia-noite apresentaram níveis significativamente mais altos de ansiedade e depressão.')
print('Com médias de GAD-7 e PHQ-9 consideravelmente maiores do que aqueles que não usam à noite.')

In [ ]:
print(df.groupby("Late_Night_Usage")["Sleep_Duration_Hours"].mean())

In [ ]:
utiliza_sim = 5.08
utiliza_nao = 6.24
diferenca_utilizacao = utiliza_nao - utiliza_sim
percentual = (diferenca_utilizacao / utiliza_sim) * 100

print(f"Usuários que usam telas após a meia-noite dormem {diferenca_utilizacao:.2f}h a menos.")
print(f"Representando {percentual:.1f}% menos de sono em comparação aos que não usam.")

#### Conclusão

Redes Sociais e Saúde Mental — Principais Achados

Realizei a análise de 8.000 registros de um conjunto de dados sintético do Kaggle, porém cientificamente fundamentado, o intuito seria verificar se o uso de telas tem impacto na saúde mental do usuário.

Identifiquei que a forma e a intensidade do uso de telas estão associadas a piores indicadores de saúde mental. Usuários classificados como Hyper-Connected (Hiperconectados) e Passive Scroller (Usuários Passivos) apresentaram mediana de ansiedade (GAD-7) de 10, contra 4 dos Digital Minimalists (Usuários Minimalistas) — uma diferença de 150%. 

Em relação ao sono, Digital Minimalists dormem em média 6,9h contra 4,7h dos Hyper-Connected. Usuários que acessam telas após a meia-noite dormem cerca de 22,8% menos e apresentam níveis significativamente maiores de ansiedade e depressão.

Plataformas de conteúdo passivo como Instagram e TikTok apresentaram mediana de ansiedade 14% acima das demais. Usuários com gatilho de comparação social ativo, que seriam usuários que consideraram que o conteúdo consumido causa inveja ou insegurança, têm média de ansiedade 43% maior.

Gênero e faixa etária (18–22 anos) não apresentaram diferença significativa, sugerindo que os padrões são transversais a esses grupos.

Importante: as associações encontradas são correlacionais — correlação não implica causalidade, e outros fatores podem contribuir para os desfechos observados.

Link do conjunto de dados utilizado do Kaggle: https://www.kaggle.com/datasets/bertnardomariouskono/social-media-and-mental-health

Link do meu Github: https://github.com/Gahbrielms

Link do meu Linkedin: https://www.linkedin.com/in/gabriel-magalh%C3%A3es-160529323/
